# 05a — LLM as Feature Extractor
**Domain:** Healthcare / Synthetic Clinical Notes &nbsp;|&nbsp; **Time:** ~2 h  
**Key Concepts:** embeddings as features, XGBoost, AUC comparison

> ⚠️ **All data is synthetic** — no real patient records are used anywhere.

---
### Approach
1. Generate synthetic clinical notes with an LLM (50 high-risk, 50 routine)
2. Embed each note → feature matrix (n, 384) with MiniLM
3. Train three XGBoost models: **tabular only | embeddings only | combined**
4. Compare AUC with 5-fold cross-validation


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
import importlib
for pkg in ["sklearn", "xgboost", "sentence_transformers"]:
    found = importlib.util.find_spec(pkg) is not None
    print(f'{"✅" if found else "❌"} {pkg}')


---
## 🔵 Example — Ex 05a-1: Synthetic Note Generation + Embedding

In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import numpy as np

client  = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def generate_note(label: int) -> str:
    condition = "high-risk cardiac event" if label == 1 else "routine annual checkup"
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user", "content":
            f"Write a 3-sentence synthetic clinical note for a patient presenting with {condition}. "
            "Include realistic but fictional vitals, symptoms, and a brief plan."}],
        max_tokens=110,
    )
    return resp.choices[0].message.content.strip()

# Quick demo — 6 notes
demo_notes  = [(generate_note(i % 2), i % 2) for i in range(6)]
demo_texts  = [n for n, _ in demo_notes]
demo_embeds = embedder.encode(demo_texts)
print(f"Embedding shape: {demo_embeds.shape}")   # (6, 384)
for note, label in demo_notes[:2]:
    print(f"[Label={label}] {note[:120]}...\n")


---
## 🟡 Exercise — Ex 05a-2: XGBoost AUC Comparison

In [ ]:
show_exercise(
    "05a-2", "XGBoost AUC comparison — 3 feature configurations",
    """Generate 100 synthetic notes (50 label=1, 50 label=0).
Create 3 tabular features per note: age (int), heart_rate (int), blood_pressure (int).
Train 3 XGBoost models with 5-fold CV:
  1. tabular only  (shape 100×3)
  2. embeddings only (shape 100×384)
  3. combined       (shape 100×387)
Report AUC for each. Which configuration wins?""",
    hints=[
        "np.random.default_rng(42) for reproducible synthetic features",
        "cross_val_score(model, X, y, cv=5, scoring='roc_auc')",
        "np.hstack([X_tab, X_emb]) to combine feature matrices",
    ],
    checks=[
        "100 notes generated (50 per class)",
        "X_emb.shape == (100, 384)",
        "X_tab.shape == (100, 3)",
        "results dict has 3 keys with AUC values > 0.5",
    ],
)


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

rng = np.random.default_rng(42)
N = 50   # per class

# TODO: generate 100 notes using generate_note()
notes, labels = [], []

# TODO: embed all notes
# X_emb = embedder.encode(notes)   # (100, 384)
X_emb = None

# TODO: create tabular features with realistic ranges
# age in [30, 80], heart_rate in [55, 120], blood_pressure in [80, 180]
X_tab = None

# TODO: combine
X_combined = None

labels_arr = np.array(labels) if labels else np.array([])

# TODO: train & evaluate 3 models
results = {}   # {"tabular": auc, "embeddings": auc, "combined": auc}


In [ ]:
check([
    (len(notes) == 100,                              f"100 notes generated (got {len(notes)})"),
    (X_emb is not None and X_emb.shape == (100, 384), "X_emb shape == (100, 384)"),
    (X_tab is not None and X_tab.shape == (100, 3),   "X_tab shape == (100, 3)"),
    (len(results) == 3,                              f"3 AUC results (got {len(results)})"),
    (all(v > 0.5 for v in results.values()),         "All AUC > 0.5 (better than random)"),
], exercise_id="05a-2")

print("\n📊 AUC Results:")
for name, auc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:15s}: {auc:.3f}")


In [ ]:
evaluate(lambda x: x,
         "Generate 100 synthetic clinical notes, embed, train 3 XGBoost models, compare AUC.",
         exercise_id="05a-2")
progress()
